# 02 — Preprocessing Pipeline
**SmartBiz AI V1.0 — Phase 3: AI Research Pipeline**

This notebook demonstrates the full preprocessing pipeline:
1. Data cleaning (missing values, duplicates, date conversion).
2. Feature engineering (calendar, lag, and rolling features).
3. Data transformation (aggregation, filtering, train/test split).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print('Libraries loaded.')

## 1. Load Raw Data

In [ ]:
from src.data.loader import load_csv

df_raw = load_csv('../data/raw/sample_sales_data.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

## 2. Data Cleaning

In [ ]:
from src.preprocessing.cleaning import clean_sales_data

df_clean = clean_sales_data(df_raw)
print(f'Cleaned shape: {df_clean.shape}')
print(f'Date dtype: {df_clean["date"].dtype}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
df_clean.head()

### 2.1 Demonstrate Missing-Value Handling
Artificially inject missing values to show the cleaning pipeline at work.

In [ ]:
# Inject some NaN values for demonstration
df_dirty = df_raw.copy()
df_dirty.loc[3, 'sales_qty'] = np.nan
df_dirty.loc[7, 'price'] = np.nan
df_dirty.loc[15, 'sales_qty'] = np.nan
print(f'Missing values (injected): {df_dirty.isnull().sum().sum()}')

from src.preprocessing.cleaning import handle_missing_values, convert_date_column

df_fixed = convert_date_column(df_dirty)
df_fixed = handle_missing_values(df_fixed, strategy='interpolate')
print(f'Missing values (after cleaning): {df_fixed.isnull().sum().sum()}')
df_fixed.iloc[[3, 7, 15]]

## 3. Feature Engineering

In [ ]:
from src.preprocessing.feature_engineering import engineer_features
from src.preprocessing.transformation import filter_product

# Filter to a single product for clarity
df_p1 = filter_product(df_clean, product_id=1)
print(f'Product 1 rows: {len(df_p1)}')

df_feat = engineer_features(df_p1)
print(f'Columns after feature engineering: {list(df_feat.columns)}')
df_feat

### 3.1 Verify Engineered Features

In [ ]:
expected_features = ['day_of_week', 'month', 'weekend_flag', 'promotion_flag',
                     'lag_1', 'lag_7', 'rolling_mean_7']

for feat in expected_features:
    status = '✅' if feat in df_feat.columns else '❌'
    print(f'{status} {feat}')

print(f'\nSample values for lag_1:\n{df_feat[["date", "sales_qty", "lag_1"]].head(5)}')
print(f'\nSample values for rolling_mean_7:\n{df_feat[["date", "sales_qty", "rolling_mean_7"]].head(8)}')

## 4. Train/Test Split

In [ ]:
from src.preprocessing.transformation import split_train_test

train_df, test_df = split_train_test(df_feat, test_ratio=0.2)

print(f'Train: {len(train_df)} rows  ({train_df["date"].min()} → {train_df["date"].max()})')
print(f'Test:  {len(test_df)} rows  ({test_df["date"].min()} → {test_df["date"].max()})')

## 5. Save Processed Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df_feat.to_csv('../data/processed/product_1_features.csv', index=False)
print('Processed data saved to data/processed/product_1_features.csv')

## Summary
- Cleaning pipeline successfully handles missing values, duplicates, and date conversion.
- All 7 required engineered features are generated correctly:
  `day_of_week`, `month`, `weekend_flag`, `promotion_flag`, `lag_1`, `lag_7`, `rolling_mean_7`.
- Temporal train/test split preserves chronological order.
- Processed data exported for baseline experiments.